# Diagnostic 3 mesures — où est le déficit du Phase 6 dual-path ?

**Objectif** : trancher scientifiquement entre 3 hypothèses pour le F1@p99 = 0.175 (vs noncausal 0.550, V5 0.512).

## Les 3 mesures (validées par l'équipe de 5 experts)

| # | Mesure | Critère verdict |
|---|--------|-----------------|
| **A** | F1@p99 du μ_HR seul (sans diffusion), pour μ_A et μ_total | F1(μ_total) ≥ 0.30 → Stage 1 capture les extrêmes |
| **B** | Pearson conditionnel ρ(pred, target \| target > p99) vs ρ_global | ρ_p99 ≪ ρ_global → info perdue dans Stage 2 |
| **C** | Eval BS30 allégée avec μ_A seul (patch) vs μ_total | F1@p99(μ_A) ≈ 0.45 → bug Stage 2 mismatch (ckpt mauvais pour μ_total) |

## Hypothèses

- **H1 (Stage 2 mismatch)** : ckpt V5 trainé pour μ_A, reçoit μ_total OOD (norm 0.096 vs train 0.041, +135%). Test : C avec μ_A.
- **H2 (Stage 1 ne capture pas les extrêmes)** : Path B dual-path ne porte pas l'info p99. Test : A.
- **H3 (Stage 2 perd l'info des extrêmes)** : μ_HR contient l'info p99 mais le dénoiseur la lisse. Test : B.

Aucun training. Aucun risque. Compute total < 15 min sur A100.

In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric, cftime, h5netcdf, xbatcher, diffusers
    from omegaconf import OmegaConf
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab')

import torch
import numpy as np
from omegaconf import OmegaConf
print(f'cwd={os.getcwd()}  torch={torch.__version__}  cuda={torch.cuda.is_available()}')

In [ ]:
# === Cell 2 : Config + Pipeline + Test dataloader ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES

DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N  = DRIVE_ROOT / 'oracle_9node' / 'seed_42'
CKPT_DUALPATH = ORACLE_9N / 'epoch_best_dualpath.pth'
CKPT_STAGE2   = ORACLE_9N / 'epoch_last.pth'
SIGMA_DATA_NEW = 0.193
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = True
CONFIG.training.num_workers = 0
ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']
OmegaConf.set_struct(CONFIG, False)
_existing = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))

K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}
DATA_ROOT = Path('/content/drive/MyDrive/climate_data/data')
LR_PATH   = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH   = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static   = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean     = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std      = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'

SEQ_LEN = int(CONFIG.data.seq_len)
pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH,
    static_path=str(_static) if _static.exists() else None,
    seq_len=SEQ_LEN, baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_variables=list(CONFIG.data.lr_variables), hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else [],
    means_path=str(_mean) if _mean.exists() else None,
    stds_path=str(_std)  if _std.exists()  else None,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
test_dataset = pipeline.build_sequence_dataset(split='test', seq_len=SEQ_LEN,
                                                stride=int(CONFIG.data.stride), as_torch=True)
val_dataset  = pipeline.build_sequence_dataset(split='val', seq_len=SEQ_LEN,
                                                stride=int(CONFIG.data.stride), as_torch=True)
test_dataloader = _DataLoader(test_dataset, batch_size=1, num_workers=0, pin_memory=True,
                               collate_fn=lambda x: x, shuffle=False)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder = HeteroGraphBuilder(lr_shape=lr_shape, hr_shape=hr_shape,
                              static_dataset=pipeline.get_static_dataset(),
                              include_mid_layer=CONFIG.graph.include_mid_layer,
                              extended_9node=True)
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
_n_test = len(test_dataset) if hasattr(test_dataset, '__len__') else '?'
print(f'[Cell 2] test samples : {_n_test}  |  HR shape ({H_HR}, {W_HR})')

# Reusable batch converter (mirrors phase6_dualpath_final_validation Cell 3).
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850','q_500','q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850','w_500','w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850','500','250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u*u + v*v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None: acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    _ivt = _compute_ivt_nodes(lr0)
    dynamic_features = {}
    for nt in builder.dynamic_node_types:
        if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
        elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
        elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
        else:              dynamic_features[nt] = _ensure_2d(lr0)
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }
print('[Cell 2] convert_sample_to_batch ready')

In [ ]:
# === Cell 3 : Load Stage 1 dual-path FROZEN ===
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder, IntelligibleVariableConfig
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder

def _clean_sd(sd):
    if sd is None: return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd

def _strip_prefixes(sd):
    if sd is None: return None
    out = {}
    for k, v in sd.items():
        nk = k
        for p in ('_orig_mod.', 'module.'):
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out

def _safe_load(module, ck, keys, label):
    for key in keys:
        sd = ck.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                m, u = module.load_state_dict(sd, strict=False)
                print(f'  [{label}] loaded from "{key}" missing={len(m)} unexpected={len(u)}')
                return True
            except Exception as e:
                print(f'  [{label}] FAILED "{key}" : {e}')
    print(f'  [{label}] no valid key')
    return False

ck_s1 = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
enc_sd = _clean_sd(ck_s1.get('encoder_state_dict', {}))
seen, order = {}, []
for k in enc_sd:
    if not k.startswith('metapath_convs.'): continue
    parts = k[len('metapath_convs.'):].split('__')
    if len(parts) < 4: continue
    name, src, rel, tgt = parts[0], parts[1], parts[2], parts[3].split('.')[0]
    if name not in seen: seen[name] = (src, rel, tgt); order.append(name)
cfgs = [IntelligibleVariableConfig(name=n, meta_path=(seen[n][0], seen[n][1], seen[n][2]), pool='mean') for n in order]
encoder = IntelligibleVariableEncoder(configs=cfgs,
                                       hidden_dim=int(CONFIG.encoder.hidden_dim),
                                       conditioning_dim=int(CONFIG.encoder.conditioning_dim)).to(DEVICE)
num_vars = len(cfgs)

_probe = next(iter(test_dataset))
C_LR = _probe['lr'].shape[1]
_lr_nodes = builder.lr_grid_to_nodes(_probe['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]

rcn_cell = RCNCell(num_vars=num_vars, hidden_dim=int(CONFIG.rcn.hidden_dim),
                    driver_dim=rcn_driver_dim, reconstruction_dim=rcn_driver_dim,
                    dropout=float(CONFIG.rcn.dropout)).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(d_model=int(rh_cfg.d_model), hr_h=H_HR, hr_w=W_HR,
                                      intermediate_h=int(rh_cfg.intermediate_h),
                                      intermediate_w=int(rh_cfg.intermediate_w),
                                      n_heads=int(rh_cfg.n_heads),
                                      refine_channels=int(rh_cfg.refine_channels),
                                      output_channels=1).to(DEVICE)
dual_path = DualPathPredictor(in_channels=C_LR, base_ch=48, hr_h=H_HR, hr_w=W_HR,
                               gate_max_mean=0.40, path_b_kind='unet',
                               path_b_unet_channels=(32, 64, 128),
                               path_b_unet_lr_shape=(23, 26)).to(DEVICE)

_safe_load(encoder, ck_s1, ['encoder_state_dict'], 'encoder')
_safe_load(rcn_cell, ck_s1, ['rcn_cell_state_dict', 'rcn_state_dict'], 'rcn_cell')
_safe_load(regression_head, ck_s1, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path, ck_s1, ['dual_path_state_dict'], 'dual_path')

for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters(): p.requires_grad_(False)
    m.eval()
print(f'[Cell 3] Stage 1 frozen. num_vars={num_vars}')

In [ ]:
# === Cell 4 : Load Stage 2 ===
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

ck_s2 = torch.load(CKPT_STAGE2, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] Stage 2 ckpt keys[:8] = {sorted(ck_s2.keys())[:8]}')
print(f'[Cell 4] ckpt epoch = {ck_s2.get("epoch", "?")}')
print(f'[Cell 4] ckpt sigma_data (if present) = {ck_s2.get("sigma_data", "?")}')

UNET_KW = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KW and isinstance(UNET_KW[_k], list):
        UNET_KW[_k] = tuple(UNET_KW[_k])
UNET_KW['projection_class_embeddings_input_dim'] = num_vars * int(CONFIG.diffusion.conditioning_dim)

edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))
_probe = next(iter(val_dataset))
hr_channels = int(_probe['residual'].shape[1])

diff = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
    unet_kwargs=UNET_KW,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=False,
    conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
    edm_config=edm_cfg, causal_concat=True,
).to(DEVICE)

_sd = _strip_prefixes(ck_s2.get('diffusion_state_dict'))
m, u = diff.load_state_dict(_sd, strict=False)
print(f'[Cell 4] missing={len(m)} unexpected={len(u)}')

diff.edm_config.sigma_data = float(SIGMA_DATA_NEW)
for p in diff.parameters(): p.requires_grad_(False)
diff.eval()
print(f'[Cell 4] Stage 2 ready. sigma_data forced = {diff.edm_config.sigma_data}')

In [ ]:
# === Cell 5 : Helpers ===
@torch.no_grad()
def compute_mu_A_and_total(batch):
    """Returns (mu_A, mu_total, baseline_log, target_residual)."""
    lr_data = batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A    = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3: mu_A = mu_A.unsqueeze(0)
    lr_grid = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, mu_B, gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)
    bl = batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1: bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)
    tgt = batch['residual'][-1].to(DEVICE)
    if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
    return mu_A, mu_total, bl, tgt, mu_B, gate

def f1_at_threshold(pred, target, percentile, valid_mask):
    """F1@p where p = percentile of target across valid pixels."""
    pred  = pred.flatten()
    target= target.flatten()
    valid = valid_mask.flatten().bool()
    pv = pred[valid]; tv = target[valid]
    if tv.numel() < 100: return float('nan')
    thr = torch.quantile(tv, percentile / 100.0)
    tp = ((pv > thr) & (tv > thr)).sum().item()
    fp = ((pv > thr) & (tv <= thr)).sum().item()
    fn = ((pv <= thr) & (tv > thr)).sum().item()
    if (tp + fp) == 0 or (tp + fn) == 0: return 0.0
    prec = tp / (tp + fp); rec = tp / (tp + fn)
    if (prec + rec) == 0: return 0.0
    return 2 * prec * rec / (prec + rec)

def pearson(a, b, eps=1e-12):
    a_c = a - a.mean(); b_c = b - b.mean()
    num = (a_c * b_c).sum()
    den = torch.sqrt((a_c * a_c).sum() * (b_c * b_c).sum() + eps)
    return float((num / den).item())

def pearson_above_threshold(pred, target, valid_mask, percentile):
    """Pearson on pixels where target > percentile."""
    pred  = pred.flatten(); target = target.flatten()
    valid = valid_mask.flatten().bool()
    pv = pred[valid]; tv = target[valid]
    if tv.numel() < 100: return float('nan')
    thr = torch.quantile(tv, percentile / 100.0)
    mask_p = tv > thr
    if mask_p.sum().item() < 50: return float('nan')
    return pearson(pv[mask_p], tv[mask_p])
print('[Cell 5] helpers ready')

In [ ]:
# === Cell 6 : MEASURE A -- F1@p99 of mu_HR alone (no diffusion) ===
# Tests : does mu_A alone (or mu_total alone) capture the p99 extremes ?
# Critical because diffusion is supposed to refine -- not invent -- extremes.
import time

N_BATCHES_A = 16  # full BS30 protocol on test split

results_A = {'mu_A': {'f1_p95': [], 'f1_p99': [], 'pearson_global': [], 'pearson_p99': [],
                       'norm_mu': [], 'norm_target': []},
             'mu_total': {'f1_p95': [], 'f1_p99': [], 'pearson_global': [], 'pearson_p99': [],
                          'norm_mu': [], 'norm_target': []},
             'baseline': {'f1_p95': [], 'f1_p99': [], 'pearson_global': [], 'pearson_p99': []}}

_t0 = time.time()
_count = 0
cached_batches = []  # store for Cell 7 re-use (IterableDataset cannot be iterated twice safely)
for sample in test_dataset:
    if _count >= N_BATCHES_A: break
    batch = convert_sample_to_batch(sample, builder, DEVICE)
    cached_batches.append(batch)
    mu_A, mu_total, baseline_log, target, mu_B, gate = compute_mu_A_and_total(batch)
    valid = torch.isfinite(target)

    # mu_A as predictor (no diffusion).
    results_A['mu_A']['f1_p95'].append(f1_at_threshold(mu_A, target, 95.0, valid))
    results_A['mu_A']['f1_p99'].append(f1_at_threshold(mu_A, target, 99.0, valid))
    results_A['mu_A']['pearson_global'].append(pearson(mu_A[valid], target[valid]))
    results_A['mu_A']['pearson_p99'].append(pearson_above_threshold(mu_A, target, valid, 99.0))
    results_A['mu_A']['norm_mu'].append(float(mu_A.abs().mean().item()))
    results_A['mu_A']['norm_target'].append(float(target[valid].abs().mean().item()))

    # mu_total as predictor.
    results_A['mu_total']['f1_p95'].append(f1_at_threshold(mu_total, target, 95.0, valid))
    results_A['mu_total']['f1_p99'].append(f1_at_threshold(mu_total, target, 99.0, valid))
    results_A['mu_total']['pearson_global'].append(pearson(mu_total[valid], target[valid]))
    results_A['mu_total']['pearson_p99'].append(pearson_above_threshold(mu_total, target, valid, 99.0))
    results_A['mu_total']['norm_mu'].append(float(mu_total.abs().mean().item()))
    results_A['mu_total']['norm_target'].append(float(target[valid].abs().mean().item()))

    # Baseline (no model at all) as reference.
    zero_pred = torch.zeros_like(target)
    results_A['baseline']['f1_p95'].append(f1_at_threshold(zero_pred, target, 95.0, valid))
    results_A['baseline']['f1_p99'].append(f1_at_threshold(zero_pred, target, 99.0, valid))
    results_A['baseline']['pearson_global'].append(0.0)
    results_A['baseline']['pearson_p99'].append(0.0)

    _count += 1
    if _count % 4 == 0:
        print(f'  [Mes A] batch {_count}/{N_BATCHES_A}', flush=True)

print(f'[MEASURE A] done in {time.time()-_t0:.1f}s on {_count} test samples')
print()
print('=' * 78)
print('MEASURE A : F1 and Pearson when using ONLY mu_HR (no diffusion)')
print('=' * 78)
for key in ['mu_A', 'mu_total', 'baseline']:
    d = results_A[key]
    f95 = np.nanmean(d['f1_p95']); f99 = np.nanmean(d['f1_p99'])
    pg  = np.nanmean(d['pearson_global']); pp99 = np.nanmean(d['pearson_p99'])
    extras = ''
    if 'norm_mu' in d:
        extras = f"  |  ‖μ‖_avg = {np.mean(d['norm_mu']):.4f}  ‖target‖_avg = {np.mean(d['norm_target']):.4f}"
    print(f'  {key:10s} : F1@p95 = {f95:.4f}  F1@p99 = {f99:.4f}  '
          f'ρ_global = {pg:.4f}  ρ@p99 = {pp99:.4f}{extras}')
print()
print('NONCAUSAL F1@p99 reference : 0.5505  (target to beat)')
print('V5/causal F1@p99 reference : 0.5123')
print('Phase 6 dualpath actual    : 0.1754  (eval today with full pipeline)')
print()
print('Interpretation :')
print('  If F1@p99(mu_total) >= 0.30  -> Stage 1 dual-path CAPTURES extremes.')
print('  If F1@p99(mu_total) <  0.30  -> Stage 1 dual-path MISSES extremes.')
print('  Compare mu_A vs mu_total : if mu_total > mu_A, Path B helps (and vice versa).')

In [ ]:
# === Cell 7 : MEASURE C -- BS30 eval (light) with mu_A vs mu_total ===
# Critical : if mu_A makes F1@p99 jump to ~0.45, the Stage 2 ckpt is OOD for mu_total.
# Then the fix is to retrain Stage 2 -- not refactor architecture.
import time

N_BATCHES_C = 4   # light : 4 batches instead of 16
K_SAMPLES_C = 32  # light : K=32 instead of 64
N_STEPS_C   = 18

def _sample_once(mu_HR, baseline_log):
    out = diff.sample(
        conditioning=None,
        num_steps=N_STEPS_C,
        scheduler_type='edm_karras',
        cfg_scale=0.0,
        apply_constraints=False,
        mu_HR=mu_HR,
        baseline_log=baseline_log,
    )
    return out.residual if hasattr(out, 'residual') else out

def eval_with_mu(mu_choice):
    """mu_choice in {'A', 'total'}."""
    means = []; targets = []; mu_used_list = []
    _t0 = time.time()
    _count = 0
    for batch in cached_batches[:N_BATCHES_C]:
        mu_A, mu_total, baseline_log, target, _mu_B, _gate = compute_mu_A_and_total(batch)
        mu_used = mu_A if mu_choice == 'A' else mu_total

        samples = [ _sample_once(mu_used, baseline_log) for _ in range(K_SAMPLES_C) ]
        samples = torch.stack(samples, dim=0)
        means.append(samples.mean(dim=0))
        targets.append(target)
        mu_used_list.append(mu_used.detach())
        _count += 1
        print(f'  [C {mu_choice}] batch {_count}/{N_BATCHES_C}', flush=True)

    pred_delta  = torch.cat(means,   dim=0).cpu()
    target_full = torch.cat(targets, dim=0).cpu()
    mu_used_all = torch.cat(mu_used_list, dim=0).cpu()
    valid = torch.isfinite(target_full)

    pred_full = mu_used_all + pred_delta
    p95 = f1_at_threshold(pred_full, target_full, 95.0, valid)
    p99 = f1_at_threshold(pred_full, target_full, 99.0, valid)
    r_global = pearson(pred_full[valid], target_full[valid])
    r_p99    = pearson_above_threshold(pred_full, target_full, valid, 99.0)
    rmse = float(((pred_full - target_full)[valid]**2).mean().sqrt().item())

    print(f'[C {mu_choice}] {_count} batches in {time.time()-_t0:.1f}s')
    return {
        'mu_choice': mu_choice,
        'n_batches': _count, 'k_samples': K_SAMPLES_C, 'n_steps': N_STEPS_C,
        'F1_p95': p95, 'F1_p99': p99,
        'pearson_global': r_global, 'pearson_p99': r_p99,
        'RMSE': rmse,
        'norm_mu_avg': float(mu_used_all.abs().mean().item()),
        'norm_target_avg': float(target_full[valid].abs().mean().item()),
        'norm_pred_delta_avg': float(pred_delta.abs().mean().item()),
        'pred_delta_flat': pred_delta,   # for measure B
        'target_full_flat': target_full,
        'pred_full_flat': pred_full,
        'mu_used_flat': mu_used_all,
        'valid_flat': valid,
    }

print('=' * 78)
print(f'MEASURE C : BS30 eval (light) -- {N_BATCHES_C} batches x K={K_SAMPLES_C} x {N_STEPS_C} steps')
print('=' * 78)
print()
print('-- Eval with mu_total (matches today\'s catastrophic result) --')
res_total = eval_with_mu('total')
print()
print('-- Eval with mu_A only (Stage 2 channel-2 in-distribution) --')
res_A = eval_with_mu('A')

In [ ]:
# === Cell 8 : MEASURE B -- Conditional Pearson on p99 mask (free from Cell 7) ===
# If pearson_global >> pearson_p99, Stage 2 dilutes the extreme signal.

def report_conditional(res, label):
    pred_delta = res['pred_delta_flat']
    target_full = res['target_full_flat']
    mu = res['mu_used_flat']
    valid = res['valid_flat']

    # 1) Pearson global of mu+delta vs target
    pred_full = res['pred_full_flat']
    r_global = pearson(pred_full[valid], target_full[valid])
    r_p99    = pearson_above_threshold(pred_full, target_full, valid, 99.0)
    r_p95    = pearson_above_threshold(pred_full, target_full, valid, 95.0)

    # 2) Decompose where the extremes signal comes from
    # if target > p99, what fraction of pred_full comes from mu vs delta ?
    target_q99 = torch.quantile(target_full[valid], 0.99)
    mask_p99 = (target_full > target_q99) & valid
    if mask_p99.sum().item() < 50:
        return

    mu_at_p99    = mu[mask_p99].abs().mean().item()
    delta_at_p99 = pred_delta[mask_p99].abs().mean().item()
    full_at_p99  = pred_full[mask_p99].abs().mean().item()
    target_at_p99 = target_full[mask_p99].abs().mean().item()

    print(f'  {label} :')
    print(f'    Pearson global       = {r_global:.4f}')
    print(f'    Pearson @ target>p95 = {r_p95:.4f}')
    print(f'    Pearson @ target>p99 = {r_p99:.4f}')
    print(f'    On p99 pixels :')
    print(f'      ‖mu‖_avg          = {mu_at_p99:.4f}')
    print(f'      ‖delta_pred‖_avg  = {delta_at_p99:.4f}')
    print(f'      ‖pred_full‖_avg   = {full_at_p99:.4f}')
    print(f'      ‖target‖_avg      = {target_at_p99:.4f}')
    print(f'      pred_full / target = {full_at_p99 / target_at_p99:.4f}  (1.0 = perfect amplitude)')
    print(f'      delta / mu        = {delta_at_p99 / max(mu_at_p99, 1e-12):.4f}  (0 = copy-mode, >>1 = δ porteur)')

print('=' * 78)
print('MEASURE B : Pearson conditioned on extremes  (free from Cell 7 results)')
print('=' * 78)
print()
report_conditional(res_total, 'eval with mu_total')
print()
report_conditional(res_A, 'eval with mu_A only')

In [ ]:
# === Cell 9 : SYNTHESE + VERDICT ===
import json

print('=' * 92)
print('SYNTHESE FINALE')
print('=' * 92)
print()
print(f'{"Predictor":<22}{"F1@p95":>10}{"F1@p99":>10}{"ρ_global":>12}{"ρ_p99":>10}{"RMSE":>10}')
print('-' * 92)

rows = []

f95_muA   = float(np.nanmean(results_A['mu_A']['f1_p95']))
f99_muA   = float(np.nanmean(results_A['mu_A']['f1_p99']))
pg_muA    = float(np.nanmean(results_A['mu_A']['pearson_global']))
pp99_muA  = float(np.nanmean(results_A['mu_A']['pearson_p99']))
f95_muT   = float(np.nanmean(results_A['mu_total']['f1_p95']))
f99_muT   = float(np.nanmean(results_A['mu_total']['f1_p99']))
pg_muT    = float(np.nanmean(results_A['mu_total']['pearson_global']))
pp99_muT  = float(np.nanmean(results_A['mu_total']['pearson_p99']))

print(f'{"mu_A alone (no diff)":<22}{f95_muA:>10.4f}{f99_muA:>10.4f}{pg_muA:>12.4f}{pp99_muA:>10.4f}{"--":>10}')
print(f'{"mu_total alone":<22}{f95_muT:>10.4f}{f99_muT:>10.4f}{pg_muT:>12.4f}{pp99_muT:>10.4f}{"--":>10}')
print(f'{"mu_A + diffusion":<22}{res_A["F1_p95"]:>10.4f}{res_A["F1_p99"]:>10.4f}'
      f'{res_A["pearson_global"]:>12.4f}{res_A["pearson_p99"]:>10.4f}{res_A["RMSE"]:>10.4f}')
print(f'{"mu_total + diffusion":<22}{res_total["F1_p95"]:>10.4f}{res_total["F1_p99"]:>10.4f}'
      f'{res_total["pearson_global"]:>12.4f}{res_total["pearson_p99"]:>10.4f}{res_total["RMSE"]:>10.4f}')
print()
print(f'References (from production runs) :')
print(f'  noncausal v4 BS30 full         F1@p99 = 0.5505  ρ_global = 0.819')
print(f'  V5 / causal BS30 full          F1@p99 = 0.5123  ρ_global = 0.825')
print(f'  Phase 6 dualpath BS30 today    F1@p99 = 0.1754  ρ_global = 0.719')
print()
print('=' * 92)
print('VERDICT')
print('=' * 92)
verdicts = []

# H1 : Stage 2 mismatch (ckpt OOD on mu_total)
if res_A['F1_p99'] > res_total['F1_p99'] + 0.10:
    verdicts.append('H1 CONFIRMED : Stage 2 ckpt is OOD for mu_total -- '
                    f'F1@p99 = {res_A["F1_p99"]:.3f} with mu_A vs {res_total["F1_p99"]:.3f} with mu_total. '
                    'Retraining Stage 2 on mu_total cache should recover ~V5 level.')
else:
    verdicts.append('H1 REJECTED : swapping mu_total -> mu_A does NOT improve F1@p99 enough '
                    f'({res_A["F1_p99"]:.3f} vs {res_total["F1_p99"]:.3f}). '
                    'Stage 2 mismatch is NOT the main cause.')

# H2 : Stage 1 misses extremes
if f99_muT >= 0.30:
    verdicts.append(f'H2 REJECTED : Stage 1 dual-path CAPTURES extremes (mu_total F1@p99 = {f99_muT:.3f} >= 0.30). '
                    'Refactoring Stage 1 is NOT needed for extremes.')
else:
    verdicts.append(f'H2 PARTIAL : Stage 1 dual-path UNDER-CAPTURES extremes (mu_total F1@p99 = {f99_muT:.3f} < 0.30). '
                    'Adding physical features (CAPE, theta_e, w700) may be warranted.')

# H3 : Stage 2 dilutes extremes
drop_p99 = res_total['pearson_global'] - res_total['pearson_p99']
if drop_p99 > 0.30:
    verdicts.append(f'H3 CONFIRMED : Stage 2 DILUTES extremes -- '
                    f'ρ_global = {res_total["pearson_global"]:.3f} vs ρ_p99 = {res_total["pearson_p99"]:.3f} (drop {drop_p99:.3f}).')
else:
    verdicts.append(f'H3 REJECTED : Stage 2 preserves extreme signal '
                    f'(ρ_global - ρ_p99 = {drop_p99:.3f} < 0.30).')

for v in verdicts:
    print(f'  - {v}')

# Save consolidated results.
OUT = ORACLE_9N / 'diagnostic_3_measures_results.json'
OUT.parent.mkdir(parents=True, exist_ok=True)
payload = {
    'measure_A_mu_alone': {
        'mu_A':    {k: [float(x) if (x == x) else None for x in v] if isinstance(v, list) else v
                    for k, v in results_A['mu_A'].items()},
        'mu_total':{k: [float(x) if (x == x) else None for x in v] if isinstance(v, list) else v
                    for k, v in results_A['mu_total'].items()},
    },
    'measure_C_BS30_light': {
        'mu_total': {k: v for k, v in res_total.items() if not torch.is_tensor(v)},
        'mu_A':     {k: v for k, v in res_A.items()     if not torch.is_tensor(v)},
    },
    'verdicts': verdicts,
    'n_batches_A': N_BATCHES_A,
    'n_batches_C': N_BATCHES_C,
    'k_samples_C': K_SAMPLES_C,
    'n_steps_C': N_STEPS_C,
}
OUT.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
print()
print(f'Saved : {OUT}')

try:
    from google.colab import files
    files.download(str(OUT))
except Exception:
    pass